# 10 Running Hermes Agent

**Purpose:** Run Nous Research's Hermes Agent in an approved Klique developer session, give it a bounded task, review its work, and stop the session.

**What we're doing**

Use a remote terminal, verify the CLI and model route, create a small practice artifact, and distinguish agent state from workspace persistence.

**Why we're doing it**

An agent connects a model to tools. The model endpoint, the agent process, and the compute hosting it are different things. Each needs a known owner, scope, and lifecycle.

**What to look for**

* The hostname and directory where tools execute
* An approved model route and limited tool access
* A small, inspectable output rather than an unattended workflow
* Saved work and a stopped compute session

**Output:** A reviewed `session_checklist.md` and notes about the environment and provider route.

This section concerns **Hermes Agent**, not a Hermes model or the ClearML worker agent. It is a terminal-based lab using Section 06 or 07. It does **not** assert that your Klique deployment has a dedicated Hermes app. Live installation, gateway compatibility, and restoration require validation in your deployment.

## 10.1 Prerequisites and Scope

Complete either [Running an SSH Session](../06_SSH/06_Running_an_SSH_Session.ipynb) or [Running VS Code](../07_VSCode/07_Running_VS_Code.ipynb). Use its launch and connection instructions, approved Linux image, project, queue, and persistent workspace. A GPU is not needed for the agent client; a separately hosted model may need one.

Have an administrator-approved Hermes installation or permission to install it, network access to approved package/model services, and provider credentials supplied through the approved secret mechanism. Use a disposable session with no production mounts, broad cloud credentials, or unrelated private files. Model calls may incur charges.

Do not connect messaging services, import an existing personal agent, install community skills/plugins, start scheduled jobs, or enable platform-management tools for this lab. A prompt is a task boundary, not a security sandbox. Tool execution remains subject to the runtime's actual permissions.

> **Checkpoint:** You know which machine will run the tools, what it can access, and who pays for the chosen model route.

> **Screenshot placeholder:** Connected SSH or VS Code terminal showing the session hostname and approved workspace, with secrets excluded.

<!-- screenshot: images/hermes_session.png (Connected terminal and verified remote hostname; mask identifiers as needed.) -->

## 10.2 Verify the Environment and CLI

Run these commands in the **remote session terminal**, not in a notebook cell or your laptop's shell:

```bash
hostname
pwd
python3 --version
git --version
command -v hermes
```

If Hermes is already installed, inspect its supported commands:

```bash
hermes --help
hermes chat --help
```

If it is missing, prefer an administrator-maintained image. Otherwise follow the official [installation guide](https://hermes-agent.nousresearch.com/docs/getting-started/installation). For an approved Linux source install, download the official installer for inspection rather than piping it directly into a shell:

```bash
curl -fsSL https://hermes-agent.nousresearch.com/install.sh -o /tmp/hermes-install.sh
less /tmp/hermes-install.sh
# Only after review and approval:
bash /tmp/hermes-install.sh
```

This is not a pinned/reproducible image build. Record the installed release or source revision for later troubleshooting. Follow the installer's shell-refresh instructions, then repeat `command -v hermes` and the help checks. Stop if installation requires unapproved system changes. Do not run as root just to get past an error.

> **Checkpoint:** The CLI is available in this remote terminal. Installation success alone does not establish model access.

## 10.3 Select and Verify the Model Route

In the remote terminal, use the official [quickstart](https://hermes-agent.nousresearch.com/docs/getting-started/quickstart) and your organization's provisioning instructions:

```bash
hermes model
```

Choose the approved provider. A custom OpenAI-compatible endpoint requires its exact base URL, model identifier, and authentication details. Enter secrets through the approved interactive/provisioning flow, never in a notebook, prompt, screenshot, or Git-tracked file. Do not enroll in a new subscription or switch to a personal key to bypass a policy rejection.

Record non-secret notes: provider, route, model, session ID, and workspace path. Hermes settings/state normally live under `~/.hermes/` on Linux, with `HERMES_HOME` available to select a different data directory. This is separate from the repository's root `.env`; Hermes does not automatically interpret this course's `KLIQUE_*` variables. See [configuration](https://hermes-agent.nousresearch.com/docs/user-guide/configuration).

### Optional: Klique LLM Gateway route

Use a gateway route only after your administrator confirms compatibility with this installed Hermes version. The course's existing gateway client sends `X-ClearML-Profile` as well as bearer authentication to a `/v1/chat/completions` route. A custom base URL and key alone may therefore be insufficient. Verify the required profile header, full API base URL, model alias, and tool-calling support; do not guess a profile path or copy a direct vLLM endpoint and call it governed.

If the required header cannot be supplied by the approved client configuration, stop the gateway portion and record the blocker. This notebook does not provide or validate an adapter. Use an independently approved provider route only if your lab permits it, and label that run as **not verified through Klique's LLM Gateway**.

> **Checkpoint:** The intended route is explicit. No claim of gateway governance is made solely because Hermes runs on Klique compute.

## 10.4 Prepare a Disposable Practice Folder

In the remote terminal, move to the **persistent workspace path you verified in Section 06 or 07**. Do not assume a path is persistent because it is named `workspace`.

Create a uniquely named directory so a repeated run cannot overwrite prior work:

```bash
LAB_DIR="$(mktemp -d "$PWD/hermes-lab.XXXXXX")"
cd "$LAB_DIR"
cat > session_notes.txt <<'EOF'
Fictional session notes:
The SSH session is ready.
The model endpoint is a separately managed resource.
Review generated files before keeping them.
Save work to the configured persistent path.
Stop the developer session when finished.
EOF
ls -la
```

Keep this folder separate from the repository and existing agent workspaces. It contains only fictional input and should contain no credentials. Note its full path for review and cleanup.

> **Checkpoint:** `session_notes.txt` is the only lab input, and no pre-existing files were overwritten.

## 10.5 Run a Bounded Agent Task

Use the installed CLI help to confirm the toolset option. The [CLI reference](https://hermes-agent.nousresearch.com/docs/user-guide/cli) documents `--toolsets`. For this exercise, expose only the terminal toolset:

Before launching, configure your lab instance's approval policy as approved by your administrator. The [security guide](https://hermes-agent.nousresearch.com/docs/user-guide/security) documents `approvals.mode: manual`. In this dedicated lab instance, use:

```bash
hermes config set approvals.mode manual
hermes chat --toolsets terminal
```

Do not disable approval checks. Toolset restriction reduces exposed tools but does not prevent a shell from accessing everything permitted to the session user.

First ask:

> In one sentence, explain the difference between an agent process and its model endpoint. Do not use tools.

After a successful response, give this task:

> Work only in the current practice directory. Read session_notes.txt and propose a five-item checklist. Ask before executing commands. After I approve, create one new file named session_checklist.md. Do not change the input file, read credentials or parent directories, access the network, install packages, schedule jobs, send messages, or change platform resources. Explain what you changed and then stop.

Review every proposed command and approve only the intended local read/write. Reject any unrelated action. A terminal tool can write files even without a dedicated file-editing toolset. Stop after this one artifact; do not turn the exercise into an autonomous loop.

> **Checkpoint:** The plain response works, the scoped tool call succeeds, and Hermes reports one new output file.

> **Screenshot placeholder:** Bounded prompt, command approval, and completion summary without credentials or unrelated history.

<!-- screenshot: images/hermes_task.png (Scoped terminal task and reviewed command approval.) -->

## 10.6 Inspect the Result Yourself

Exit Hermes, then inspect the directory from the same remote terminal:

```bash
pwd
ls -la
cat session_notes.txt
cat session_checklist.md
```

Compare the checklist with the fictional notes. It should contain five actionable items, mention saving work and stopping the developer session, and distinguish the model endpoint's lifecycle. No new installation or platform change was requested. Missing output is a failed checkpoint, even if the agent said it completed the task.

Record what you observed and correct mistakes deliberately. Do not execute generated shell commands just because they appear in a Markdown checklist.

For a **verified gateway-routed run**, inspect the corresponding Klique AI Usage time window and profile. Confirm actual usage rather than assuming one user prompt equals one model request. For another provider route, use its permitted usage view and do not claim Klique gateway quotas applied.

> **Checkpoint:** The output exists, its contents match the input, and the usage route is described accurately.

## 10.7 Preserve State and Stop

Keep only the reviewed artifact and non-secret lab notes as a source export. Agent history, memory, skills, and credentials are not interchangeable with that artifact. Treat the Hermes data directory and workspace snapshots as potentially sensitive. Do not copy the entire agent home into this repository or a shared archive.

Record the persistence policy for both the practice directory and Hermes state. A fresh container may restore files without restoring installed packages or a running process. After a later restore, check the hostname, CLI availability, route, and artifact again before continuing; restoration has not been validated by this notebook.

Exit the agent and use the SSH/VS Code lesson's Klique **Stop** procedure. Verify the instance is stopped. Closing a terminal is not sufficient. Do not stop a shared model endpoint merely because this client exercise is finished.

> **Checkpoint:** Your reviewed work is saved and the developer session is stopped.

> **Screenshot placeholder:** Stopped Klique instance and non-secret record of the saved practice artifact.

<!-- screenshot: images/hermes_stop.png (Stopped developer session and artifact location.) -->

## Troubleshooting

| Symptom | What to check |
|---|---|
| `hermes` not found | Confirm remote hostname, approved installation, and refreshed shell PATH. |
| Installation fails | Inspect the installer log and image/network prerequisites; do not use root or bypass restrictions as a shortcut. |
| Authentication or model error | Confirm exact provider, endpoint, model identifier, and credential scope with the administrator. |
| Chat works but tools fail | Check model tool-calling support, selected toolset, approvals, and filesystem permissions. |
| Gateway rejects the request | Verify profile header, alias, route, and quota. Do not retry indefinitely or evade governance. |
| Extra files or unexpected commands | Stop the agent, inspect the practice directory, and narrow the task. |
| Output disappears on restore | Verify configured persistence paths and separately verify Hermes installation/state. |
| Usage absent from Klique AI Usage | Confirm whether the call actually passed through the LLM Gateway. Hosting alone is not proof. |

## Wrap-Up

You ran Hermes Agent as a client on managed compute, reviewed a small tool-produced artifact, and separated model routing from tool permissions and workspace lifecycle.

**Validation boundary:** This is a guided Markdown walkthrough. Course schema/link checks do not install Hermes, call a model, verify a Klique-native Hermes app, or test a live gateway/session restore. Capture those deployment checks before presenting the lab as production-validated.

**Back:** [Running Claude Desktop](../09_ClaudeDesktop/09_Running_Claude_Desktop.ipynb)  
**Index:** [README](../README.md)
